✅ Fonction simplifiée créée!


In [5]:
file_path = "documents/test/Specification.md"

In [8]:
file = Path(file_path).read_text(encoding="utf-8")
file

"## 1- Contexte et Architecture Legacy\nL'architecture v1 du pipeline d'ingestion s'articule autour de trois composants distincts : l'Ingesteur HTTP (nommé Alpha), le Moteur de Règles (Beta), et le Stockage Persistant (Gamma). Lors de l'incident d'exploitation du 14 janvier, une surcharge réseau a provoqué une indisponibilité globale de 3 heures. L'analyse post-mortem a établi que la défaillance initiale ne provenait ni du point d'entrée principal ni de la base de données, mais uniquement du composant intermédiaire chargé de l'évaluation synchrone.\n\n## 2- Correctifs d'Urgence et Effets de Bord\nCe composant intermédiaire a saturé sa mémoire tampon suite à une montée en charge dépassant 1 500 req/s. Pour stabiliser le service sans refondre l'infrastructure, le patch v2.4.1 a été appliqué en urgence sur cet élément précis. Cependant, cette modification a introduit un effet de bord critique : une latence systématique de 45 ms a été ajoutée sur chaque écriture effectuée par le composant 

In [6]:
chunks = chunk_markdown_by_title(file_path)

In [7]:
chunks

[{'file_name': 'Specification.md',
  'file_path': 'documents/test/Specification.md',
  'content': "## 1- Contexte et Architecture Legacy\n\nL'architecture v1 du pipeline d'ingestion s'articule autour de trois composants distincts : l'Ingesteur HTTP (nommé Alpha), le Moteur de Règles (Beta), et le Stockage Persistant (Gamma). Lors de l'incident d'exploitation du 14 janvier, une surcharge réseau a provoqué une indisponibilité globale de 3 heures. L'analyse post-mortem a établi que la défaillance initiale ne provenait ni du point d'entrée principal ni de la base de données, mais uniquement du composant intermédiaire chargé de l'évaluation synchrone."},
 {'file_name': 'Specification.md',
  'file_path': 'documents/test/Specification.md',
  'content': "## 2- Correctifs d'Urgence et Effets de Bord\n\nCe composant intermédiaire a saturé sa mémoire tampon suite à une montée en charge dépassant 1 500 req/s. Pour stabiliser le service sans refondre l'infrastructure, le patch v2.4.1 a été appliqué

In [19]:
from llama_cpp import Llama
import llama_cpp.llama_chat_format

In [24]:
llm = Llama(
    model_path="./models/Qwen3-1.7B-Q8_0.gguf",
    n_ctx=32768,
    n_gpu_layers=-1,
    n_threads=8,
    verbose=False
)

In [29]:
print(llm.metadata.get("tokenizer.chat_template"))

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}
{%- for index in range(ns.last_query_index, -1, -1) %}
    {%- set message =

In [ ]:
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "Tu es un assistant utile et concis."},
        {"role": "user", "content": "Explique la photosynthèse en 3 phrases."},
    ],
    max_tokens=256,
    temperature=0.7,
)

print(response["choices"][0]["message"]["content"])

In [30]:
import re

# Ajoutez ici les paires (ouverture, fermeture) selon les modèles que vous utilisez
KNOWN_TAG_PAIRS = [
    ("<think>", "</think>"),
    ("[THINK]", "[/THINK]"),
    ("<thinking>", "</thinking>"),
    ("<reasoning>", "</reasoning>"),
]

def strip_thinking(text: str, tag_pairs=KNOWN_TAG_PAIRS) -> str:
    for open_tag, close_tag in tag_pairs:
        pattern = re.compile(re.escape(open_tag) + r".*?" + re.escape(close_tag), re.DOTALL)
        if open_tag in text:
            text = pattern.sub("", text)
        elif close_tag in text:  # tag ouvrant absent (déjà injecté par le prompt)
            text = re.sub(r"^.*?" + re.escape(close_tag), "", text, flags=re.DOTALL)
    return text.strip()

In [31]:
strip_thinking(response["choices"][0]["message"]["content"])

"La photosynthèse est un processus par lequel les plantes utilisent la lumière solaire, l'azote du dioxyde de carbone et l'eau pour produire de l'énergie sous forme de glucose et de l'oxygène. Les chlorophylles, pigments chlorophytiques des plantes, captent la lumière et catalysent la réaction chimique"

In [23]:
response

{'id': 'chatcmpl-372e3a04-11c9-4072-847d-f5718841e599',
 'object': 'chat.completion',
 'created': 1788222100,
 'model': './models/Qwen3-1.7B-Q8_0.gguf',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': "La photosynthèse est le processus par which les plantes, les algues et certains microorganismes transforment la lumière solaire en énergie chimique. Elle se produit dans les chloroplastes des cellules vertes. Les plantes utilisent l'air respiré (dioxygène) et l'eau pour produire du glucose et émettre du dioxygène."},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 39, 'completion_tokens': 84, 'total_tokens': 123}}

In [32]:
llm.close
del llm

In [8]:
import pandas as pd

chunks = pd.read_csv("data/BRIGHT/biology/context_chunks.csv")
chunks = chunks.to_dict(orient="records")

len(chunks)


5684